In [ ]:
from pathlib import Path
import os
import json
import re
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
import pathlib
import numpy as np
import pandas as pd
from scipy.signal import spectrogram, decimate
from scipy.ndimage import zoom
import pywt
from html import escape
from time import monotonic
from IPython.display import HTML, display

In [24]:
Wibracje_Folder = pathlib.Path.cwd().parent.parent

In [ ]:
DATASET_MODE = "both"  # "collected", "public", "both"

COLLECTED_RAW_DIR = Wibracje_Folder / "Datasety" / "Wibracje wyważanie" / "processed_full"
PUBLIC_RAW_DIR = Wibracje_Folder / "Datasety" / "Machine_Fault_Data" / "imbalance"

FOLDER_LEVELS_UP_BEFORE_PREPROCESSING = 2
PREPROCESSING_FOLDER_NAME = "data_preprocessing"
COLLECTED_DATASET_NAME = "collected"
PUBLIC_DATASET_NAME = "public"
RAW_FEATURES_DIR_NAME = "Raw_Features"

CLEAR_RAW_FEATURES = True

FEATURE_GENERATION_WORKERS = 8

COLLECTED_FS = 32768.0
PUBLIC_FS = 50000.0

TIME_COL = "Time_s"
COLLECTED_SIGNAL_COLS = ["ACXXX_SN1 [g]", "ACXXX_SN2 [g]"]
COLLECTED_TACHO_COL = "Tacho_Ch8 [Hz]"
PUBLIC_SENSOR_LIMIT = 2

F_MIN_HZ = 1.0
F_MAX_HZ = 500.0
SPEED_MIN_HZ = 5.0
SPEED_MAX_HZ = 100.0

NPERSEG = 2 ** 15
NOVERLAP_RATIO = 0.75

USE_CWT = True
CWT_FREQUENCY_BINS = 128
CWT_WAVELET = "cmor1.5-1.0"
COLLECTED_CWT_DOWNSAMPLE_FACTOR = 50
PUBLIC_CWT_DOWNSAMPLE_FACTOR = 26

ANGLE_DIVISIONS = 17
PUBLIC_ANGLE_LABEL = 0
PUBLIC_RADIUS_MM = 150.0

R_ZEW_MM = 100.0
R_SR_MM = 90.0
COLLECTED_LOCATION_TO_IMBALANCE_GMM = {
    "baseline": 0.0,
    "sr": 1.79 * R_SR_MM,
    "srodek": 1.79 * R_SR_MM,
    "środek": 1.79 * R_SR_MM,
    "zew": 1.79 * R_ZEW_MM,
    "zewnatrz": 1.79 * R_ZEW_MM,
    "zewnątrz": 1.79 * R_ZEW_MM,
    "zew2": 2.61 * R_ZEW_MM,
    "zew_sr": (1.79 * R_ZEW_MM) + (1.75 * R_SR_MM),
    "zew_srodek": (1.79 * R_ZEW_MM) + (1.75 * R_SR_MM),
    "zew_środek": (1.79 * R_ZEW_MM) + (1.75 * R_SR_MM),
    "zewd3": R_ZEW_MM * (2.42 + 3 * 0.76), 
    "zewd2": R_ZEW_MM * (2.42 + 2 * 0.76), 
    "zewdd": R_ZEW_MM * (2.42 + 2 * 0.76), 
    "zew3":  R_ZEW_MM * 3.37, 
}

PREPROCESSING_BASE_DIR = Path.cwd()
for _ in range(FOLDER_LEVELS_UP_BEFORE_PREPROCESSING):
    PREPROCESSING_BASE_DIR = PREPROCESSING_BASE_DIR.parent

PREPROCESSING_ROOT = PREPROCESSING_BASE_DIR / PREPROCESSING_FOLDER_NAME
print("PREPROCESSING_ROOT:", PREPROCESSING_ROOT)

PREPROCESSING_ROOT: c:\Users\szymo\Desktop\Wibracje\data_preprocessing


In [26]:
2.42 + 3 * 0.76

4.7

In [27]:
"""
info do nowych oznaczen:
zew - sruba wkrecona w dziure zewnetrzna o wadze 1.79g
sr - sruba wkrecona w dziure srodkowa o wadze 1.79g
zew sr - sruba wkrecana w dziure zewnetrzna o wadze 1.79g, oraz sruba o wadze 1.75g wkrecona w dziure srodkowa
zew2 - sruba z naktretka wkrecona w dziure zewnetrzna o wadze 2.61g
a1 - kat w ktorym byla wkrecona sruba
zewd3 - duza sruba z 2 nakretkami od tacho i 1 od czujnika w dziure zewnetrzna o wadze 4.7g
zewd2 - duza sruba z 1 nakretka od tacho i 1 od czujnika w dziure zewnetrzna o wadze 3.94g
zewdd - duza sruba z 2 nakretkami od tacho w dziure zewnetrzna o wadze 3.94g
zew3 - sruba z 2 nakretkami wkrecona w dziure zewnetrzna o wadze 3.37g
zew2 2.61g (1.85g + 0.76g)
zew 1.79g i sr
zew sr (tylko sr 1.75)
nakretka 0.76g
duz 2.42
mala 1.79
srednia 1.85
zewd3 (2.42g + 3 * 0.76g)
zewd2 (2.42g + 2 * 0.76g)
zewdd (2.42g + 2 * 0.76g)
zew3 3.37g
"""

'\ninfo do nowych oznaczen:\nzew - sruba wkrecona w dziure zewnetrzna o wadze 1.79g\nsr - sruba wkrecona w dziure srodkowa o wadze 1.79g\nzew sr - sruba wkrecana w dziure zewnetrzna o wadze 1.79g, oraz sruba o wadze 1.75g wkrecona w dziure srodkowa\nzew2 - sruba z naktretka wkrecona w dziure zewnetrzna o wadze 2.61g\na1 - kat w ktorym byla wkrecona sruba\nzewd3 - duza sruba z 2 nakretkami od tacho i 1 od czujnika w dziure zewnetrzna o wadze 4.7g\nzewd2 - duza sruba z 1 nakretka od tacho i 1 od czujnika w dziure zewnetrzna o wadze 3.94g\nzewdd - duza sruba z 2 nakretkami od tacho w dziure zewnetrzna o wadze 3.94g\nzew3 - sruba z 2 nakretkami wkrecona w dziure zewnetrzna o wadze 3.37g\nzew2 2.61g (1.85g + 0.76g)\nzew 1.79g i sr\nzew sr (tylko sr 1.75)\nnakretka 0.76g\nduz 2.42\nmala 1.79\nsrednia 1.85\nzewd3 (2.42g + 3 * 0.76g)\nzewd2 (2.42g + 2 * 0.76g)\nzewdd (2.42g + 2 * 0.76g)\nzew3 3.37g\n'

In [28]:
def dataset_names_from_mode(mode):
    mode = str(mode).lower()
    if mode == "both":
        return [COLLECTED_DATASET_NAME, PUBLIC_DATASET_NAME]
    if mode in {COLLECTED_DATASET_NAME, PUBLIC_DATASET_NAME}:
        return [mode]
    raise ValueError("DATASET_MODE must be 'collected', 'public', or 'both'.")


def natural_key(path):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", Path(path).name)]


def clean_token(value):
    text = str(value)
    for old in [" ", "/", "\\", ":", ";", ",", "."]:
        text = text.replace(old, "_")
    return text


def clean_signal_1d(x):
    x = np.asarray(x, dtype=np.float64)
    if x.size == 0:
        return x
    finite = np.isfinite(x)
    if finite.all():
        return x
    if not finite.any():
        return np.zeros_like(x, dtype=np.float64)
    idx = np.arange(x.size)
    x = x.copy()
    x[~finite] = np.interp(idx[~finite], idx[finite], x[finite])
    return x


def normalize_location_text(text):
    if text is None:
        return None
    return str(text).lower().replace(" ", "_").replace("-", "_")


def angle_label_to_values(angle_label):
    angle_label = int(angle_label)
    angle_rad = 2 * np.pi * angle_label / ANGLE_DIVISIONS
    return {
        "angle_label": angle_label,
        "angle_deg": float(np.degrees(angle_rad) % 360),
        "angle_rad": float(angle_rad),
        "angle_sin_cos": np.array([np.sin(angle_rad), np.cos(angle_rad)], dtype=np.float32),
    }


def relative_from_dataset(value, min_value, max_value):
    value = float(value)
    if np.isclose(max_value, min_value):
        return 0.0
    return float((value - min_value) / (max_value - min_value))

In [ ]:
def parse_collected_metadata(file_path):
    labelled_pattern = re.compile(r"^v(?P<v_main>\d+)_(?P<v_sub>\d+)_a(?P<angle>\d+)_(?P<location>zew_sr|zew2|zew|sr|zewd3|zewd2|zewdd|zew3)_processed\.csv$", re.IGNORECASE,)
    baseline_pattern = re.compile(r"^v(?P<v_main>\d+)_(?P<v_sub>\d+)_processed\.csv$", re.IGNORECASE,)


    match = labelled_pattern.match(file_path.name)
    if match:
        groups = match.groupdict()
        location = normalize_location_text(groups["location"])
        return {
            "dataset": COLLECTED_DATASET_NAME,
            "path": file_path,
            "source_file": file_path.name,
            "source_stem": file_path.stem,
            "v_main": int(groups["v_main"]),
            "v_sub": int(groups["v_sub"]),
            "angle_label": int(groups["angle"]),
            "location": location,
            "is_baseline": False,
            "imbalance_actual_gmm": float(COLLECTED_LOCATION_TO_IMBALANCE_GMM[location]),
        }

    match = baseline_pattern.match(file_path.name)
    if match:
        groups = match.groupdict()
        return {
            "dataset": COLLECTED_DATASET_NAME,
            "path": file_path,
            "source_file": file_path.name,
            "source_stem": file_path.stem,
            "v_main": int(groups["v_main"]),
            "v_sub": int(groups["v_sub"]),
            "angle_label": 0,
            "location": "baseline",
            "is_baseline": True,
            "imbalance_actual_gmm": 0.0,
        }

    return None


def build_collected_catalog(raw_dir):
    rows = []
    ignored = []
    unsupported = []
    for path in sorted(Path(raw_dir).glob("*.csv"), key=natural_key):
        if not path.name.endswith("_processed.csv"):
            ignored.append(path)
            continue
        meta = parse_collected_metadata(path)
        if meta is None:
            unsupported.append(path)
        else:
            rows.append(meta)
    if unsupported:
        print("Unsupported collected files:")
        for path in unsupported[:20]:
            print(" ", path.name)
        print(f" {len(unsupported)} collected *_processed.csv files do not match the known patterns.")
    catalog = pd.DataFrame(rows)
    if len(catalog) > 0:
        vmin = float(catalog["imbalance_actual_gmm"].min())
        vmax = float(catalog["imbalance_actual_gmm"].max())
        catalog["imbalance_relative"] = catalog["imbalance_actual_gmm"].apply(lambda x: relative_from_dataset(x, vmin, vmax))
    print(f"Collected files used: {len(catalog)} | ignored: {len(ignored)}")
    return catalog


def parse_public_mass_g(file_path):
    return float(str(file_path.parent.name).lower().replace("g", ""))


def build_public_catalog(raw_dir):
    rows = []
    for path in sorted(Path(raw_dir).rglob("*.csv"), key=lambda p: (p.parent.name, natural_key(p))):
        mass_g = parse_public_mass_g(path)
        rows.append({
            "dataset": PUBLIC_DATASET_NAME,
            "path": path,
            "source_file": path.name,
            "source_stem": path.stem,
            "relative_path": str(path.relative_to(raw_dir)),
            "mass_g": float(mass_g),
            "radius_mm": float(PUBLIC_RADIUS_MM),
            "angle_label": int(PUBLIC_ANGLE_LABEL),
            "location": "public_fixed_radius",
            "is_baseline": False,
            "imbalance_actual_gmm": float(mass_g * PUBLIC_RADIUS_MM),
        })
    catalog = pd.DataFrame(rows)
    if len(catalog) > 0:
        vmin = float(catalog["imbalance_actual_gmm"].min())
        vmax = float(catalog["imbalance_actual_gmm"].max())
        catalog["imbalance_relative"] = catalog["imbalance_actual_gmm"].apply(lambda x: relative_from_dataset(x, vmin, vmax))
    print(f"Public files used: {len(catalog)}")
    return catalog

In [30]:
def read_collected_file(path):
    cols = [TIME_COL] + COLLECTED_SIGNAL_COLS + [COLLECTED_TACHO_COL]
    df = pd.read_csv(path, usecols=cols)
    for col in cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    signals = {
        "Sensor_1": df[COLLECTED_SIGNAL_COLS[0]].to_numpy(),
        "Sensor_2": df[COLLECTED_SIGNAL_COLS[1]].to_numpy(),
    }
    tacho = df[COLLECTED_TACHO_COL].to_numpy()
    finite_tacho = tacho[np.isfinite(tacho)]
    has_tacho = bool(len(finite_tacho) > 0 and np.any(np.abs(finite_tacho) > 0))
    if has_tacho:
        signals["Tachometer"] = tacho
    return df, signals, has_tacho


def read_public_file(path):
    df = pd.read_csv(path, header=None)
    sensor_count = min(PUBLIC_SENSOR_LIMIT, df.shape[1])
    signals = {}
    for i in range(sensor_count):
        signals[f"Sensor_{i + 1}"] = pd.to_numeric(df.iloc[:, i], errors="coerce").to_numpy()
    return df, signals, False


def compute_complex_spectrogram(x, fs):
    x = clean_signal_1d(x)
    nperseg = min(int(NPERSEG), len(x))
    noverlap = min(int(NOVERLAP_RATIO * nperseg), max(0, nperseg - 1))
    f, t, z = spectrogram(
        x,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        detrend="constant",
        scaling="spectrum",
        mode="complex",
    )
    mask = (f >= F_MIN_HZ) & (f <= F_MAX_HZ)
    return f[mask].astype(np.float32), t.astype(np.float32), z[mask, :]


def complex_to_maps(z):
    amp = 20 * np.log10(np.abs(z) + np.finfo(float).eps)
    phase = np.angle(z)
    return {
        "amplitude_db": amp.astype(np.float32),
        "phase_sin": np.sin(phase).astype(np.float32),
        "phase_cos": np.cos(phase).astype(np.float32),
    }


def resize_2d(arr, target_shape):
    arr = np.asarray(arr, dtype=np.float32)
    target_shape = tuple(int(v) for v in target_shape)
    if arr.shape == target_shape:
        return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    factors = (target_shape[0] / arr.shape[0], target_shape[1] / arr.shape[1])
    resized = zoom(arr, factors, order=1)
    out = np.zeros(target_shape, dtype=np.float32)
    h = min(target_shape[0], resized.shape[0])
    w = min(target_shape[1], resized.shape[1])
    out[:h, :w] = resized[:h, :w]
    if h < target_shape[0] and h > 0:
        out[h:, :w] = out[h - 1:h, :w]
    if w < target_shape[1] and w > 0:
        out[:, w:] = out[:, w - 1:w]
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def compute_cwt_amplitude_db(x, fs, downsample_factor, target_shape):
    x = clean_signal_1d(x)
    x = x - np.mean(x)
    factor = max(1, int(downsample_factor))
    fs_eff = float(fs)
    if factor > 1:
        x = decimate(x, factor, zero_phase=True)
        fs_eff = float(fs) / factor
    freqs = np.geomspace(max(F_MIN_HZ, 1e-6), F_MAX_HZ, int(CWT_FREQUENCY_BINS))
    scales = pywt.central_frequency(CWT_WAVELET) * fs_eff / freqs
    coeffs, _ = pywt.cwt(x, scales, CWT_WAVELET, sampling_period=1 / fs_eff)
    cwt_amp = 20 * np.log10(np.abs(coeffs) + np.finfo(float).eps)
    return resize_2d(cwt_amp, target_shape)

In [ ]:
def estimate_speed_from_fft(signals, fs):
    peaks = []
    for signal in signals.values():
        x = clean_signal_1d(signal)
        if len(x) < 2:
            continue
        x = x - np.mean(x)
        freqs = np.fft.rfftfreq(len(x), d=1 / fs)
        amp = np.abs(np.fft.rfft(x))
        mask = (freqs >= SPEED_MIN_HZ) & (freqs <= SPEED_MAX_HZ)
        if np.any(mask):
            local_freqs = freqs[mask]
            local_amp = amp[mask]
            peaks.append(float(local_freqs[int(np.argmax(local_amp))]))
    if not peaks:
        return 0.0
    return float(np.median(peaks))


def speed_from_collected_tacho(df, target_time_s):
    source_time = pd.to_numeric(df[TIME_COL], errors="coerce").to_numpy(dtype=np.float64)
    speed = pd.to_numeric(df[COLLECTED_TACHO_COL], errors="coerce").to_numpy(dtype=np.float64)
    valid = np.isfinite(source_time) & np.isfinite(speed) & (speed >= SPEED_MIN_HZ) & (speed <= SPEED_MAX_HZ)
    if valid.sum() >= 2:
        order = np.argsort(source_time[valid])
        return np.interp(target_time_s, source_time[valid][order], speed[valid][order]).astype(np.float32), "tachometer"
    return None, "tachometer_invalid"


def build_speed_map(dataset_name, df, signals, has_tacho, fs, target_time_s, target_shape):
    if dataset_name == COLLECTED_DATASET_NAME and has_tacho:
        speed_t, source = speed_from_collected_tacho(df, target_time_s)
        if speed_t is not None:
            return np.tile(speed_t.reshape(1, -1), (target_shape[0], 1)).astype(np.float32), source, float(np.nanmedian(speed_t))
    speed_hz = estimate_speed_from_fft({k: v for k, v in signals.items() if k != "Tachometer"}, fs)
    speed_t = np.full(len(target_time_s), speed_hz, dtype=np.float32)
    return np.tile(speed_t.reshape(1, -1), (target_shape[0], 1)).astype(np.float32), "fft", float(speed_hz)


def add_tachometer_phase_difference_maps(channel_features):
    tach_maps = channel_features.get("Tachometer")
    if tach_maps is None:
        return

    for channel_name, maps in channel_features.items():
        if not str(channel_name).startswith("Sensor_"):
            continue

        sin_tacho = tach_maps["phase_sin"]
        cos_tacho = tach_maps["phase_cos"]

        if sin_tacho.shape != maps["phase_sin"].shape:
            sin_tacho = resize_2d(sin_tacho, maps["phase_sin"].shape)
        if cos_tacho.shape != maps["phase_cos"].shape:
            cos_tacho = resize_2d(cos_tacho, maps["phase_cos"].shape)

        maps["phase_sin_tacho_minus_sensor"] = np.nan_to_num(sin_tacho - maps["phase_sin"], nan=0.0, posinf=0.0, neginf=0.0,
        ).astype(np.float32)
        maps["phase_cos_tacho_minus_sensor"] = np.nan_to_num(cos_tacho - maps["phase_cos"], nan=0.0, posinf=0.0, neginf=0.0 ).astype(np.float32)


def stack_feature_maps(dataset_name, df, signals, has_tacho, fs, cwt_downsample_factor):
    channel_features = {}
    frequency_hz = None
    time_s = None
    target_shape = None

    for channel_name, signal in signals.items():
        f, t, z = compute_complex_spectrogram(signal, fs)
        maps = complex_to_maps(z)
        maps["cwt_amplitude_db"] = compute_cwt_amplitude_db(signal, fs, cwt_downsample_factor, maps["amplitude_db"].shape) if USE_CWT else np.zeros_like(maps["amplitude_db"])
        channel_features[channel_name] = maps
        if frequency_hz is None:
            frequency_hz = f
            time_s = t
            target_shape = maps["amplitude_db"].shape

    if has_tacho:
        add_tachometer_phase_difference_maps(channel_features)

    speed_map, speed_source, speed_hz = build_speed_map(dataset_name, df, signals, has_tacho, fs, time_s, target_shape)

    component_names = []
    arrays = []
    sensor_order = ["Sensor_1", "Sensor_2"]
    channel_order = list(sensor_order)
    if has_tacho:
        channel_order.append("Tachometer")

    base_feature_keys = ["amplitude_db", "phase_sin", "phase_cos", "cwt_amplitude_db"]
    tacho_difference_keys = ["phase_sin_tacho_minus_sensor", "phase_cos_tacho_minus_sensor"]

    for channel_name in channel_order:
        if channel_name not in channel_features:
            continue
        for key in base_feature_keys:
            arrays.append(channel_features[channel_name][key].astype(np.float32))
            component_names.append(f"{channel_name} | {key}")

    if has_tacho and "Tachometer" in channel_features:
        for sensor_name in sensor_order:
            if sensor_name not in channel_features:
                continue
            for key in tacho_difference_keys:
                if key not in channel_features[sensor_name]:
                    continue
                arrays.append(channel_features[sensor_name][key].astype(np.float32))
                component_names.append(f"Tachometer | {sensor_name} | {key}")

    arrays.append(speed_map.astype(np.float32))
    component_names.append("speed_hz_map")

    X = np.stack(arrays, axis=0).astype(np.float32)
    return X, component_names, frequency_hz.astype(np.float32), time_s.astype(np.float32), speed_source, speed_hz


In [ ]:
def make_labels(row, dataset_min, dataset_max):
    angle = angle_label_to_values(row["angle_label"])
    actual = float(row["imbalance_actual_gmm"])
    return {
        "angle_label": int(angle["angle_label"]),
        "angle_deg": float(angle["angle_deg"]),
        "angle_rad": float(angle["angle_rad"]),
        "angle_sin_cos": angle["angle_sin_cos"],
        "imbalance_actual_gmm": actual,
        "imbalance_relative": relative_from_dataset(actual, dataset_min, dataset_max),
        "mass_g": None if "mass_g" not in row or pd.isna(row.get("mass_g")) else float(row["mass_g"]),
        "radius_mm": None if "radius_mm" not in row or pd.isna(row.get("radius_mm")) else float(row["radius_mm"]),
    }


def output_file_name(row):
    if row["dataset"] == COLLECTED_DATASET_NAME:
        return f"{row['source_stem']}_features.npy"
    mass = clean_token(f"{row['mass_g']:g}g")
    stem = clean_token(row["source_stem"])
    return f"public_{mass}_{stem}_features.npy"


def save_label_scaling(dataset_root, catalog):
    values = catalog["imbalance_actual_gmm"].astype(float)
    info = {
        "imbalance_actual_unit": "g*mm",
        "relative_formula": "(actual - dataset_min) / (dataset_max - dataset_min)",
        "dataset_min_gmm": float(values.min()) if len(values) else 0.0,
        "dataset_max_gmm": float(values.max()) if len(values) else 0.0,
        "angle_divisions": int(ANGLE_DIVISIONS),
    }
    with open(dataset_root / "label_scaling.json", "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, ensure_ascii=False)
    return info


def process_feature_row(row, dataset_name, dataset_min, dataset_max, label_scaling, out_dir):
    row = dict(row)
    path = Path(row["path"])

    try:
        if dataset_name == COLLECTED_DATASET_NAME:
            df, signals, has_tacho = read_collected_file(path)
            fs = COLLECTED_FS
            cwt_factor = COLLECTED_CWT_DOWNSAMPLE_FACTOR
        else:
            df, signals, has_tacho = read_public_file(path)
            fs = PUBLIC_FS
            cwt_factor = PUBLIC_CWT_DOWNSAMPLE_FACTOR

        X, component_names, frequency_hz, time_s, speed_source, speed_hz = stack_feature_maps(
            dataset_name=dataset_name,
            df=df,
            signals=signals,
            has_tacho=has_tacho,
            fs=fs,
            cwt_downsample_factor=cwt_factor,
        )

        labels = make_labels(row, dataset_min, dataset_max)
        metadata = {
            "dataset": dataset_name,
            "source_file": row["source_file"],
            "source_stem": row["source_stem"],
            "source_path": str(path),
            "source_has_tachometer": bool(has_tacho),
            "has_tachometer": bool(has_tacho),
            "fs": float(fs),
            "speed_source": speed_source,
            "speed_hz": float(speed_hz),
            "location": row.get("location", None),
            "is_baseline": bool(row.get("is_baseline", False)),
        }

        for optional_key in ["v_main", "v_sub", "relative_path"]:
            if optional_key in row and not pd.isna(row[optional_key]):
                metadata[optional_key] = row[optional_key]

        payload = {
            "X": X,
            "component_names": component_names,
            "frequency_hz": frequency_hz,
            "time_s": time_s,
            "labels": labels,
            "metadata": metadata,
            "feature_keys": ["amplitude_db", "phase_sin", "phase_cos", "cwt_amplitude_db", "phase_sin_tacho_minus_sensor", "phase_cos_tacho_minus_sensor", "speed_hz_map"],
            "label_scaling": label_scaling,
        }

        out_path = out_dir / output_file_name(row)
        np.save(out_path, payload, allow_pickle=True)
        return out_path

    except Exception as exc:
        raise RuntimeError(f"Failed while processing {path}") from exc


class NotebookProgressBar:
    def __init__(self, total, description="Progress", min_interval_seconds=0.25):
        self.total = max(int(total), 0)
        self.description = str(description)
        self.min_interval_seconds = float(min_interval_seconds)
        self.completed = 0
        self.started_at = monotonic()
        self.last_render_at = 0.0
        self.handle = None

    def _elapsed_text(self):
        elapsed = max(monotonic() - self.started_at, 0.0)
        if elapsed < 60:
            return f"{elapsed:.1f}s"
        minutes, seconds = divmod(int(elapsed), 60)
        return f"{minutes}m {seconds}s"

    def _render_html(self, status="running"):
        pct = 100.0 if self.total == 0 else 100.0 * self.completed / self.total
        pct = max(0.0, min(100.0, pct))
        safe_description = escape(self.description)
        safe_status = escape(status)
        return f"""
        <div style="font-family: system-ui, -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; min-width: 360px;">
            <div style="display: flex; justify-content: space-between; gap: 1rem; margin-bottom: 0.25rem;">
                <strong>{safe_description}</strong>
                <span>{self.completed}/{self.total} ({pct:.1f}%)</span>
            </div>
            <progress value="{self.completed}" max="{max(self.total, 1)}" style="width: 100%; height: 1.2rem;"></progress>
            <div style="font-size: 0.9em; opacity: 0.75; margin-top: 0.25rem;">
                Status: {safe_status} · elapsed: {self._elapsed_text()}
            </div>
        </div>
        """

    def start(self):
        self.update(0, force=True)

    def update(self, completed=None, force=False, status="running"):
        if completed is not None:
            self.completed = int(completed)
        now = monotonic()
        if not force and self.completed < self.total and (now - self.last_render_at) < self.min_interval_seconds:
            return
        self.last_render_at = now

        if HTML is not None and display is not None:
            html = HTML(self._render_html(status=status))
            if self.handle is None:
                self.handle = display(html, display_id=True)
            else:
                self.handle.update(html)
        else:
            pct = 100.0 if self.total == 0 else 100.0 * self.completed / self.total
            print(f"\r{self.description}: {self.completed}/{self.total} ({pct:.1f}%)", end="", flush=True)
            if self.completed >= self.total:
                print()

    def close(self):
        self.update(self.completed, force=True, status="complete")


def process_dataset(dataset_name, catalog):
    dataset_root = PREPROCESSING_ROOT / dataset_name
    out_dir = dataset_root / RAW_FEATURES_DIR_NAME
    if CLEAR_RAW_FEATURES and out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if len(catalog) == 0:
        print(f"No files for dataset: {dataset_name}")
        return []

    label_scaling = save_label_scaling(dataset_root, catalog)
    dataset_min = label_scaling["dataset_min_gmm"]
    dataset_max = label_scaling["dataset_max_gmm"]

    rows = catalog.to_dict("records")
    workers = max(1, int(FEATURE_GENERATION_WORKERS))
    workers = min(workers, len(rows))

    saved = []
    print(f"Generating {dataset_name} with {workers} worker(s).")

    progress = NotebookProgressBar(total=len(rows), description=f"Generating {dataset_name}")
    progress.start()
    completed = 0

    try:
        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = [
                executor.submit(
                    process_feature_row,
                    row=row,
                    dataset_name=dataset_name,
                    dataset_min=dataset_min,
                    dataset_max=dataset_max,
                    label_scaling=label_scaling,
                    out_dir=out_dir,
                )
                for row in rows
            ]

            for future in as_completed(futures):
                saved.append(future.result())
                completed += 1
                progress.update(completed)
    finally:
        progress.close()

    saved = sorted(saved, key=natural_key)
    manifest = pd.DataFrame({"path": [str(p) for p in saved], "file": [p.name for p in saved]})
    manifest.to_csv(dataset_root / "raw_features_manifest.csv", index=False)
    print(f"Saved {len(saved)} files to: {out_dir}")
    return saved

In [33]:
selected_datasets = dataset_names_from_mode(DATASET_MODE)

all_saved = {}
for dataset_name in selected_datasets:
    if dataset_name == COLLECTED_DATASET_NAME:
        catalog = build_collected_catalog(COLLECTED_RAW_DIR)
    else:
        catalog = build_public_catalog(PUBLIC_RAW_DIR)

    display(catalog.head())
    if len(catalog):
        print("Imbalance actual range [g*mm]:", float(catalog["imbalance_actual_gmm"].min()), float(catalog["imbalance_actual_gmm"].max()))
        print("Angle labels:")
        display(catalog["angle_label"].value_counts(dropna=False).sort_index())

    all_saved[dataset_name] = process_dataset(dataset_name, catalog)

print("Done.")
for dataset_name, paths in all_saved.items():
    print(dataset_name, len(paths))

Unsupported collected files:
  v08_14_a7_zew_d3_processed.csv
 1 collected *_processed.csv files do not match the known patterns.
Collected files used: 585 | ignored: 0


,dataset,path,source_file,source_stem,v_main,v_sub,angle_label,location,is_baseline,imbalance_actual_gmm,imbalance_relative
0,collected,c:\Users\szymo\Desktop\Wibracje\Datasety\Wibra...,v01_14_a1_sr_processed.csv,v01_14_a1_sr_processed,1,14,1,sr,False,161.1,0.342766
1,collected,c:\Users\szymo\Desktop\Wibracje\Datasety\Wibra...,v01_14_a1_zew2_processed.csv,v01_14_a1_zew2_processed,1,14,1,zew2,False,261.0,0.555319
2,collected,c:\Users\szymo\Desktop\Wibracje\Datasety\Wibra...,v01_14_a1_zew3_processed.csv,v01_14_a1_zew3_processed,1,14,1,zew3,False,337.0,0.717021
3,collected,c:\Users\szymo\Desktop\Wibracje\Datasety\Wibra...,v01_14_a1_zew_processed.csv,v01_14_a1_zew_processed,1,14,1,zew,False,179.0,0.380851
4,collected,c:\Users\szymo\Desktop\Wibracje\Datasety\Wibra...,v01_14_a1_zewd2_processed.csv,v01_14_a1_zewd2_processed,1,14,1,zewd2,False,394.0,0.838298


Imbalance actual range [g*mm]: 0.0 470.0
Angle labels:


angle_label
0     14
1     81
3     78
5     78
7     77
11    95
13    81
15    81
Name: count, dtype: int64

Generating collected with 8 worker(s).


...

,dataset,path,source_file,source_stem,relative_path,mass_g,radius_mm,angle_label,location,is_baseline,imbalance_actual_gmm,imbalance_relative
0,public,c:\Users\szymo\Desktop\Wibracje\Datasety\Machi...,12.288.csv,12.288,0g\12.288.csv,0.0,150.0,0,public_fixed_radius,False,0.0,0.0
1,public,c:\Users\szymo\Desktop\Wibracje\Datasety\Machi...,13.1072.csv,13.1072,0g\13.1072.csv,0.0,150.0,0,public_fixed_radius,False,0.0,0.0
2,public,c:\Users\szymo\Desktop\Wibracje\Datasety\Machi...,14.336.csv,14.336,0g\14.336.csv,0.0,150.0,0,public_fixed_radius,False,0.0,0.0
3,public,c:\Users\szymo\Desktop\Wibracje\Datasety\Machi...,15.1552.csv,15.1552,0g\15.1552.csv,0.0,150.0,0,public_fixed_radius,False,0.0,0.0
4,public,c:\Users\szymo\Desktop\Wibracje\Datasety\Machi...,16.1792.csv,16.1792,0g\16.1792.csv,0.0,150.0,0,public_fixed_radius,False,0.0,0.0


Imbalance actual range [g*mm]: 0.0 5250.0
Angle labels:


angle_label
0    382
Name: count, dtype: int64

Generating public with 8 worker(s).


Saved 382 files to: c:\Users\szymo\Desktop\Wibracje\data_preprocessing\public\Raw_Features
Done.
collected 585
public 382
